# 09 — Exit Analysis & NLP
**Goal**: Understand why employees leave through termination text analysis

**ML Progression**: Text preprocessing → LDA Topic Modeling → VADER Sentiment → Combined Clustering

**HR Value**: Deeper exit interview insights, retention program targeting

**Employee Value**: Voice is heard and analyzed for change

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, re
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from nltk.sentiment.vader import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/09_exit_nlp'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} terminated employees')

## 1. Text Preprocessing & Overview

In [ ]:
print(f'Records with text: {df["cleaned_desc"].str.len().gt(0).sum()}')
print(f'Avg word count: {df["word_count"].mean():.1f}')
print(f'\nSample descriptions:')
for i in range(min(5, len(df))):
    print(f'  [{df.iloc[i]["TerminationType"]}] {df.iloc[i]["TerminationDescription"][:100]}')

## 2. LDA Topic Modeling

In [ ]:
texts = df['cleaned_desc'].fillna('').tolist()
vectorizer = CountVectorizer(max_features=500, stop_words='english', min_df=5)
dtm = vectorizer.fit_transform(texts)

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(dtm)

feature_names = vectorizer.get_feature_names_out()
print('LDA Topics:')
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f'  Topic {topic_idx + 1}: {", ".join(top_words)}')

In [ ]:
df['topic'] = lda.transform(dtm).argmax(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
df['topic'].value_counts().sort_index().plot(kind='bar', ax=ax, title='Topic Distribution')
ax.set_xlabel('Topic')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/09_topic_distribution.png', bbox_inches='tight')
plt.show()

## 3. VADER Sentiment Analysis

In [ ]:
import nltk
nltk.download('vader_lexicon', quiet=True)
sid = SentimentIntensityAnalyzer()
df['sentiment'] = df['cleaned_desc'].fillna('').apply(
    lambda x: sid.polarity_scores(x)['compound'])
df['sentiment_label'] = pd.cut(
    df['sentiment'], bins=[-1, -0.05, 0.05, 1],
    labels=['Negative', 'Neutral', 'Positive'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['sentiment'].dropna(), bins=30)
axes[0].set_title('Sentiment Score Distribution')
df['sentiment_label'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Sentiment Labels')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/09_sentiment.png', bbox_inches='tight')
plt.show()

## 4. Exit Patterns by Termination Type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pd.crosstab(df['topic'], df['TerminationType'], normalize='index').plot(
    kind='bar', stacked=True, ax=ax, title='Topic Composition by Termination Type')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/09_topic_by_type.png', bbox_inches='tight')
plt.show()

## 5. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. {n_topics} exit topics identified from termination descriptions')
print(f'2. Sentiment distribution: {df["sentiment_label"].value_counts().to_dict()}')
print()
print('--- HR Action Items ---')
print('- Target retention programs at patterns from negative sentiment exits')
print('- Monitor topic trends over time for early warning signs')
print()
print('--- Employee Impact ---')
print('- Exit voice is systematically analyzed for improvement')
print('- Themes inform workplace culture changes')